In [6]:
#!/usr/bin/env python3
"""
Run all Train / Testing ML experiments from the manuscript:

- Timepoints: M1 (preop), M2 (end of surgery), M3 (POD2 / within 48h labs)
- Architectures: Logistic Regression, Random Forest, XGBoost
- Evaluation: 5-fold Stratified CV with scaling + SMOTE on training folds only
- Outputs: CSVs for per-fold metrics, out-of-fold predictions, calibration by procedure cluster (M2),
           SHAP values and global importances (M2 XGB by default),
           duration quantile features (global + per-chopcluster) with per-fold cutpoints,
           and saved train/test feature tables for M2 *before SMOTE* for downstream association analysis.

This script is based on the provided notebook codebase and keeps its data-loading contract:
  ./data/study_meta.csv
  ./data/study_TSF.csv
  ./data/study_labs2.csv
  ./data/selectedmarkers.csv
  ./data/study_com.csv
  ./data/study_proc.csv

"""
from __future__ import annotations

import os
import json
import argparse
import warnings
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, RobustScaler, PowerTransformer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, confusion_matrix, brier_score_loss
)
from imblearn.over_sampling import SMOTE

from xgboost import XGBClassifier

# SHAP (optional)
try:
    import shap  # type: ignore
    _HAS_SHAP = True
except Exception:
    shap = None
    _HAS_SHAP = False

warnings.filterwarnings("ignore")
pd.options.display.max_columns = 500

In [10]:
################
# HELPERS
def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

def save_run_manifest(outdir: str) -> None:
    import platform
    import sklearn
    try:
        import xgboost
    except Exception:
        xgboost = None
    try:
        import imblearn
    except Exception:
        imblearn = None

    manifest = {
        "python": sys_version(),
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "sklearn": sklearn.__version__,
        "xgboost": getattr(xgboost, "__version__", None),
        "imblearn": getattr(imblearn, "__version__", None),
        "shap": getattr(shap, "__version__", None) if _HAS_SHAP else None,
    }
    path = os.path.join(outdir, "run_manifest.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

def sys_version() -> str:
    import sys
    return sys.version.replace("\n", " ")

################
# CORE UTILITIES
def clean_dataframe(df: pd.DataFrame) -> Tuple[pd.DataFrame, List[str]]:
    """Remove constant columns and return clean df + numeric col list."""
    to_remove: List[str] = []
    num_cols: List[str] = []
    for c in df.columns:
        if df[c].nunique(dropna=False) <= 1:
            to_remove.append(c)
        else:
            if pd.api.types.is_numeric_dtype(df[c]):
                num_cols.append(c)
    return df.drop(columns=to_remove), num_cols


class Imputer:
    """Predictive imputer for lab values."""
    def __init__(self, target: str, cols: List[str], model, value_range: Tuple[float, float]):
        self.target = target
        self.cols = cols
        self.model = model
        self.value_range = value_range

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        if self.target not in df.columns:
            df[self.target] = np.nan

        mask_missing = df[self.target].isnull()

        for c in self.cols:
            if c not in df.columns:
                df[c] = np.nan

        if mask_missing.any():
            X_pred = df.loc[mask_missing, self.cols].fillna(0)
            preds = self.model.predict(X_pred)
            preds = np.clip(preds, self.value_range[0], self.value_range[1])
            df.loc[mask_missing, self.target] = preds
        return df


def load_and_preprocess_data(base_path: str, day_x: int, _id: str) -> Tuple[pd.DataFrame, pd.DataFrame, List[str], List[Tuple[str, int]]]:
    """
    Loads and merges base cohort and returns:
      df_final: merged modelling cohort with base (non-lab) features + endpoint
      dfl: long lab table with 'POD_value' column
      selected_markers: list of markers to impute/include
      forbidden_imputation: list of (marker, POD) pairs to skip
    """
    inputs_path = os.path.join(base_path, "data")
    #meta_path = os.path.join(base_path, "meta")

    dfl = pd.read_csv(os.path.join(inputs_path, "study_labs2.csv"))
    dfl["POD_value"] = dfl.apply(lambda x: f"{x['marker']}_{x['t']}", axis=1)

    dfm = pd.read_csv(os.path.join(inputs_path,  "study_meta.csv"))
    dfc = pd.read_csv(os.path.join(inputs_path,  "study_com.csv"))
    dfch = pd.read_csv(os.path.join(inputs_path, "study_proc.csv"))
    dfv = pd.read_csv(os.path.join(inputs_path,  "study_TSF.csv"))

    ##!TOBIAS
    if {'chopcluster_10', 'chopcluster_9'}.issubset(dfch.columns):
        dfch['chopcluster_10'] = dfch['chopcluster_10']+dfch['chopcluster_9']
        dfch = dfch.drop(columns=['chopcluster_9'])

    imput_sel = pd.read_csv(os.path.join(inputs_path, "selectedmarkers.csv"))
    selected_markers = sorted(set(imput_sel[imput_sel["include"] == 1].ft))
    forbidden_imputation = list(
        imput_sel[imput_sel["include"] == 0][["ft", "POD"]].itertuples(index=False, name=None)
    )

    # Filtering meta (same as notebook)
    dfm = dfm[~dfm["clinic"].isin(["dermatology", "plastic_hand", "ophthalmology"])]
    dfm = dfm[dfm["asa"] <= 4]

    ##!TOBIAS added:
    dfm = dfm[dfm["duration_of_surg"] >= 30]
    
    if "inf_parasitic" in dfm.columns:
        dfm = dfm.drop(columns=["inf_parasitic"])

    if {'diagnosis', 'chop_op'}.issubset(dfm.columns):
        dfm = dfm.dropna(subset=["diagnosis", "chop_op"])
    
    dfm["emergencysurgery"] = dfm["emergencysurgery"].fillna(0).astype(int)

    # Merge
    dfm = pd.merge(dfm, dfc, on=_id)
    dfm = pd.merge(dfm, dfch, on=_id)
    dfm = pd.merge(dfm, dfv, on=_id)

    # Feature groups (same as notebook)
    cols_pre = ["age", "sex", "asa"]
    cols_com = [x for x in dfm.columns if x.startswith("com_")]
    
    ##!TOBIAS changed:
    #cols_intra = ["emergencysurgery", "atb_preop", "night_surg","imcib", "duration_of_surg"]
    cols_intra = ["emergencysurgery", "atb_preop", "night_surg"]
    cols_chop = [x for x in dfch.columns if x.startswith("chopcluster_")]
    cols_periop = [x for x in dfv.columns if x != _id]
    
    ##!TOBIAS changed:
    #cols_postop = ["com_elix_Coagulopathy"]
    cols_postop = ["com_elix_Coagulopathy", "imcib", "duration_of_surg"]
    
    features_t2 = cols_pre + cols_com + cols_intra + cols_periop + cols_chop + cols_postop

    # Cohort restriction (kept from notebook: LOS >= day_x)
    dx = dfm[dfm["length_of_stay"] >= day_x][[_id]]

    endpoint = "class"
    df_model = dfm[[_id, endpoint] + features_t2].copy()
    df_model = df_model[df_model[endpoint] != -1]
    df_final = pd.merge(dx, df_model, on=_id)

    # Drop specific comorbidities (notebook did in a later cell)
    drop_cols = [c for c in ["com_elix_FluidEcletrolyteDisorders", "com_elix_Coagulopathy"] if c in df_final.columns]
    if drop_cols:
        df_final = df_final.drop(columns=drop_cols)

    print('>>>FINAL COHORT:',len(df_final))
    return df_final, dfl, selected_markers, forbidden_imputation


def train_lab_imputers(
    X_train_full: pd.DataFrame,
    dfl: pd.DataFrame,
    markers: List[str],
    forbidden: List[Tuple[str, int]],
    day_x: int,
    _id: str,
    random_state: int
) -> Dict[Tuple[str, int], Imputer]:
    """Train per-fold lab imputers on training IDs only."""
    from sklearn.experimental import enable_hist_gradient_boosting  # noqa
    from sklearn.ensemble import HistGradientBoostingRegressor

    dfl_train = dfl[
        (dfl["marker"].isin(markers)) &
        (dfl["t"] >= 0) & (dfl["t"] <= day_x) &
        (dfl[_id].isin(X_train_full[_id]))
    ]
    pivot_train = pd.pivot(dfl_train, index=_id, columns="POD_value", values="value").reset_index()
    train_for_impute = pd.merge(pivot_train, X_train_full, on=_id)

    imputers: Dict[Tuple[str, int], Imputer] = {}

    for day_Z in range(0, day_x + 1):
        for m in markers:
            if (m, day_Z) in forbidden:
                continue
            target_col = f"{m}_{day_Z}"
            current_cols = [c for c in train_for_impute.columns if c not in [target_col, _id]]

            subset = train_for_impute.dropna(subset=[target_col])
            if subset.empty:
                continue

            y_imp = subset[target_col]
            X_imp = subset[current_cols].fillna(0)

            X_imp, _ = clean_dataframe(X_imp)
            clean_cols = X_imp.columns.tolist()

            model = HistGradientBoostingRegressor(random_state=random_state)
            model.fit(X_imp, y_imp)

            imputers[(m, day_Z)] = Imputer(
                target=target_col,
                cols=clean_cols,
                model=model,
                value_range=(float(y_imp.min()), float(y_imp.max()))
            )
    return imputers


def apply_lab_logic(
    base_df: pd.DataFrame,
    lab_df: pd.DataFrame,
    _id: str,
    imputers: Dict[Tuple[str, int], Imputer],
    markers: List[str],
    forbidden: List[Tuple[str, int]],
    day_x: int
) -> pd.DataFrame:
    """Pivot labs, merge, apply per-marker/per-day imputers, and add AVG_ and NA_ features (as notebook)."""
    sub_lab = lab_df[
        (lab_df["marker"].isin(markers)) &
        (lab_df["t"] >= 0) & (lab_df["t"] <= day_x) &
        (lab_df[_id].isin(base_df[_id]))
    ]

    pivot = pd.pivot(sub_lab, index=_id, columns="POD_value", values="value").reset_index()
    merged = pd.merge(base_df, pivot, on=_id, how="left")

    for day_Z in range(0, day_x + 1):
        for m in markers:
            if (m, day_Z) in forbidden:
                continue
            if (m, day_Z) in imputers:
                merged = imputers[(m, day_Z)].transform(merged)

    if day_x > 0:
        for m in markers:
            m_cols = [c for c in merged.columns if c.startswith(f"{m}_")]
            if m_cols:
                merged[f"AVG_{m}"] = merged[m_cols].mean(axis=1)

        # piv_na = pd.pivot(sub_lab, index=_id, columns="POD_value", values="value").reset_index()
        # for c in piv_na.columns:
        #     if c == _id:
        #         continue
        #     merged[f"NA_{c}"] = merged[c].apply(lambda x: 0 if pd.isnull(x) else 1)

    return merged


def add_duration_quartile_bins(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    col: str = "duration_of_surg",
    new_col: str = "duration_of_surg_qbin"
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, float]]:
    """
    Compute global quartile cutpoints on TRAIN only, apply to TRAIN and TEST.
    Bins: 0..3 (quartiles). Missing -> -1.
    """
    train_vals = pd.to_numeric(train_df.get(col, np.nan), errors="coerce")
    non_null = train_vals.dropna()

    if len(non_null) == 0:
        train_df[new_col] = -1
        test_df[new_col] = -1
        stats = {"q25": np.nan, "q50": np.nan, "q75": np.nan, "mean": np.nan, "median": np.nan}
        return train_df, test_df, stats

    q25, q50, q75 = np.percentile(non_null.values, [25, 50, 75])
    mean_v = float(non_null.mean())
    median_v = float(non_null.median())

    bins = [-np.inf, q25, q50, q75, np.inf]
    labels = [1, 2, 3, 4]

    for df_ in (train_df, test_df):
        vals = pd.to_numeric(df_.get(col, np.nan), errors="coerce")
        df_[new_col] = pd.cut(vals, bins=bins, labels=labels, include_lowest=True)
        df_[new_col] = df_[new_col].cat.add_categories([-1]).fillna(-1).astype(int)

    stats = {"q25": float(q25), "q50": float(q50), "q75": float(q75), "mean": mean_v, "median": median_v}
    return train_df, test_df, stats


def add_duration_quartile_bins_by_chopclusters(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    cluster_cols: List[str],
    duration_col: str = "duration_of_surg",
    prefix: str = "duration_of_surg_qbin__"
) -> Tuple[pd.DataFrame, pd.DataFrame, List[Dict[str, object]]]:
    """
    For each cluster col:
      - compute quartiles (train only) on rows where cluster > 0
      - assign bins to those rows (1..4); other rows -> 0
      - if in-group but duration missing -> -1
    Returns stats_rows: dicts with cutpoints per fold/cluster.
    """
    stats_rows: List[Dict[str, object]] = []

    train_dur = pd.to_numeric(train_df.get(duration_col, np.nan), errors="coerce")
    test_dur  = pd.to_numeric(test_df.get(duration_col, np.nan), errors="coerce")

    for ccol in cluster_cols:
        new_col = f"{prefix}{ccol}"

        if ccol not in train_df.columns:
            train_df[new_col] = 0
            test_df[new_col] = 0
            stats_rows.append({
                "cluster_col": ccol, "n_train_in_group": 0,
                "q25": np.nan, "q50": np.nan, "q75": np.nan,
                "mean": np.nan, "median": np.nan
            })
            continue

        train_in = pd.to_numeric(train_df[ccol], errors="coerce").fillna(0) > 0
        test_in  = pd.to_numeric(test_df[ccol], errors="coerce").fillna(0) > 0

        dur_sub = train_dur[train_in].dropna()

        train_df[new_col] = 0
        test_df[new_col] = 0

        if len(dur_sub) == 0:
            stats_rows.append({
                "cluster_col": ccol,
                "n_train_in_group": int(train_in.sum()),
                "q25": np.nan, "q50": np.nan, "q75": np.nan,
                "mean": np.nan, "median": np.nan
            })
            continue

        q25, q50, q75 = np.percentile(dur_sub.values, [25, 50, 75])
        mean_v = float(dur_sub.mean())
        median_v = float(dur_sub.median())

        bins = [-np.inf, q25, q50, q75, np.inf]
        labels = [1, 2, 3, 4]

        train_bins = pd.cut(train_dur, bins=bins, labels=labels, include_lowest=True)
        train_df.loc[train_in & train_dur.isna(), new_col] = -1
        train_df.loc[train_in & (~train_dur.isna()), new_col] = (
            train_bins[train_in & (~train_dur.isna())].astype(int)
        )
        train_df[new_col] = train_df[new_col].astype(int)

        test_bins = pd.cut(test_dur, bins=bins, labels=labels, include_lowest=True)
        test_df.loc[test_in & test_dur.isna(), new_col] = -1
        test_df.loc[test_in & (~test_dur.isna()), new_col] = (
            test_bins[test_in & (~test_dur.isna())].astype(int)
        )
        test_df[new_col] = test_df[new_col].astype(int)

        stats_rows.append({
            "cluster_col": ccol,
            "n_train_in_group": int(train_in.sum()),
            "q25": float(q25), "q50": float(q50), "q75": float(q75),
            "mean": mean_v, "median": median_v
        })

    return train_df, test_df, stats_rows


########################################
# Feature-set construction for M1/M2/M3
@dataclass(frozen=True)
class FeatureGroups:
    pre: List[str]
    com: List[str]
    intra: List[str]
    periop: List[str]
    chop: List[str]
    postop: List[str]


def infer_feature_groups(df: pd.DataFrame) -> FeatureGroups:
    pre = [c for c in ["age", "sex", "asa"] if c in df.columns]
    com = [c for c in df.columns if c.startswith("com_")]

    ##!TOBIAS changed:
    intra = [c for c in ["emergencysurgery", "atb_preop", "night_surg"] if c in df.columns]
    chop = [c for c in df.columns if c.startswith("chopcluster_")]
    # periop time-series derived features come from crp_feat3; in merged df they are everything else not in above buckets,
    # but we keep a conservative rule: columns containing '|' are your engineered signal features.
    periop = [c for c in df.columns if "|" in c]
    ##!TOBIAS changed:
    postop = ["imcib", "duration_of_surg"]

    return FeatureGroups(pre=pre, com=com, intra=intra, periop=periop, chop=chop, postop=postop)


def select_features(df: pd.DataFrame, groups: FeatureGroups, timepoint: str) -> List[str]:
    """
    timepoint:
      - M1: pre + com + intra <--- ##!TOBIAS added
      - M2: pre + com + intra + periop + chop + postop (no labs)
      - M3: M2 base + labs (added later by apply_lab_logic)

    ##!TOBIAS
    Split the surgery variables into two groups 
        - 'intra' variables: available beginning / during the surgery: ["emergencysurgery", "atb_preop", "night_surg"]
        - 'post' variables: available immediately after the surgery: ["imcib", "duration_of_surg"]

    """
    if timepoint == "M1":
        ##!TOBIAS changed:
        #return sorted(list(dict.fromkeys(groups.pre + groups.com)))
        return sorted(list(dict.fromkeys(groups.pre + groups.com + groups.intra)))
    if timepoint in ("M2", "M3"):
        return sorted(list(dict.fromkeys(groups.pre + groups.com + groups.intra + groups.periop + groups.chop + groups.postop)))
    raise ValueError(f"Unknown timepoint: {timepoint}")


##########
# Metrics
def compute_binary_metrics(y_true: np.ndarray, y_prob: np.ndarray, threshold: float = 0.5) -> Dict[str, float]:
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    # guard divisions
    def safe_div(a: float, b: float) -> float:
        return float(a / b) if b != 0 else float("nan")

    sens = safe_div(tp, (tp + fn))
    spec = safe_div(tn, (tn + fp))
    ppv  = safe_div(tp, (tp + fp))
    npv  = safe_div(tn, (tn + fn))

    out = {
        "auc": float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) == 2 else float("nan"),
        "sensitivity": sens,
        "specificity": spec,
        "ppv": ppv,
        "npv": npv,
        "tp": float(tp), "fp": float(fp), "tn": float(tn), "fn": float(fn),
    }
    return out


def summarize_cv_metrics(df: pd.DataFrame, group_cols: List[str]) -> pd.DataFrame:
    num_cols = [c for c in df.columns if c not in group_cols]
    agg = df.groupby(group_cols)[num_cols].agg(["mean", "std"]).reset_index()
    # flatten columns
    agg.columns = ["_".join([c for c in col if c]) for col in agg.columns.to_flat_index()]
    return agg


##########
# Modeling
def build_preprocessor(feature_names: List[str]) -> ColumnTransformer:
    """
    Uses the suffix groups from the notebook to decide scalers.
    """
    ##!TOBIAS changed
    standard_suff = ["age", "asa", "duration_of_surg", "lab|", "|mean", "|std", "|skew", "|kurtosis", "|entropy", 
                     "|trend_slope", "|trend_linregress_intercept", "|peak_interval_mean", "|trough_interval_mean"]
    robust_suff   = ["|median", "|range", "|max_depth", "|num_episodes", "|peak_count", "|trough_count"]
    power_suff    = ["|auc", "|time", "|min", "|max"]

    std_ft = [f for f in feature_names if any(s in f for s in standard_suff)]
    rob_ft = [f for f in feature_names if any(s in f for s in robust_suff)]
    pow_ft = [f for f in feature_names if any(s in f for s in power_suff)]

    return ColumnTransformer(
        transformers=[
            ("std", StandardScaler(), std_ft),
            ("robust", RobustScaler(), rob_ft),
            ("power", PowerTransformer(), pow_ft),
        ],
        remainder="passthrough"
    )


def make_model(arch: str, seed: int) -> object:
    if arch == "logreg":
        return LogisticRegression(
            max_iter=2000,
            solver="liblinear",
            random_state=seed
        )
    if arch == "rf":
        return RandomForestClassifier(
            n_estimators=200,
            max_depth=None,
            min_samples_leaf=1,
            random_state=seed,
            n_jobs=-1
        )
    if arch == "xgb":
        return XGBClassifier(
            n_estimators=200,
            learning_rate=0.1,
            reg_lambda=1.0,
            scale_pos_weight=2.0,
            random_state=seed,
            n_jobs=4,
            eval_metric="logloss"
        )
    raise ValueError(f"Unknown arch: {arch}")


#################
# Calibration
def infer_primary_chopcluster(row: pd.Series, chop_cols: List[str]) -> Optional[str]:
    vals = row[chop_cols].values.astype(float)
    # If one-hot, pick max > 0. Otherwise take first positive.
    if np.all(np.isnan(vals)) or len(chop_cols) == 0:
        return None
    mx = np.nanmax(vals)
    if mx <= 0:
        return None
    idx = int(np.nanargmax(vals))
    return chop_cols[idx]


def compute_cluster_calibration(
    oof_df: pd.DataFrame,
    chop_cols: List[str],
    n_bins: int = 10
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    oof_df must contain: _id, y_true, y_prob, plus chopcluster_* columns.
    Returns:
      - calib_points: per cluster and probability bin: mean_pred, mean_obs, n
      - cluster_summary: per cluster: brier, bias, prevalence, n
    """
    df = oof_df.copy()
    df["cluster_col"] = df.apply(lambda r: infer_primary_chopcluster(r, chop_cols), axis=1)
    df = df.dropna(subset=["cluster_col"])

    points_rows: List[Dict[str, object]] = []
    summary_rows: List[Dict[str, object]] = []

    for cluster, g in df.groupby("cluster_col"):
        y = g["y_true"].astype(int).values
        p = g["y_prob"].astype(float).values
        if len(g) == 0:
            continue

        brier = float(brier_score_loss(y, p))
        prev = float(np.mean(y))
        bias = float(np.mean(p) - prev)

        summary_rows.append({
            "cluster_col": cluster,
            "n": int(len(g)),
            "prevalence": prev,
            "brier": brier,
            "calibration_bias": bias,
        })

        # Bin by predicted-prob quantiles within cluster
        try:
            bins = pd.qcut(g["y_prob"], q=n_bins, duplicates="drop")
        except Exception:
            # fallback to equal-width bins
            bins = pd.cut(g["y_prob"], bins=n_bins)

        tmp = g.assign(prob_bin=bins)
        for b, gg in tmp.groupby("prob_bin"):
            points_rows.append({
                "cluster_col": cluster,
                "prob_bin": str(b),
                "n": int(len(gg)),
                "mean_pred": float(gg["y_prob"].mean()),
                "mean_obs": float(gg["y_true"].mean()),
                "min_pred": float(gg["y_prob"].min()),
                "max_pred": float(gg["y_prob"].max()),
            })

    calib_points = pd.DataFrame(points_rows)
    cluster_summary = pd.DataFrame(summary_rows).sort_values(["n"], ascending=False)
    return calib_points, cluster_summary


##############
# Main runner
##############

def run_all(
    base_path: str,
    outdir: str,
    day_x: int,
    n_splits: int,
    _id: str,
    seed: int,
    do_shap: bool,
    shap_timepoint: str,
    shap_arch: str,
    keep_m2_train_test: bool
) -> None:
    set_all_seeds(seed)
    os.makedirs(outdir, exist_ok=True)
    save_run_manifest(outdir)

    # Load base cohort + lab table
    print('## Loading and preprocessing data')
    df_all, dfl, markers, forbidden = load_and_preprocess_data(base_path=base_path, day_x=day_x, _id = _id)

    # Endpoint
    y_all = df_all["class"].astype(int)
    df_features_all = df_all.drop(columns=["class"])

    groups = infer_feature_groups(df_all)

    # Prepare CV
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    # For duration bin cutpoints
    duration_cutpoints_rows: List[Dict[str, object]] = []

    # Store metrics & predictions for each experiment
    metrics_rows: List[Dict[str, object]] = []
    preds_rows: List[Dict[str, object]] = []

    # SHAP accumulation
    shap_rows: List[pd.DataFrame] = []
    shap_importance_rows: List[Dict[str, object]] = []

    # M2 features for association analysis (raw, no SMOTE)
    m2_assoc_tables: List[pd.DataFrame] = []

    timepoints = ["M1", "M2", "M3"]
    archs = ["logreg", "rf", "xgb"]

    for fold, (train_idx, test_idx) in enumerate(skf.split(df_features_all, y_all), start=1):
        print('## Fold',fold)
        X_train_full = df_features_all.iloc[train_idx].copy()
        X_test_full  = df_features_all.iloc[test_idx].copy()
        y_train = y_all.iloc[train_idx].copy()
        y_test  = y_all.iloc[test_idx].copy()

        for tp in timepoints:
            print('### Model',tp)
            base_cols = select_features(df_all, groups, tp)
            # Always keep _id for joins/exports
            keep_cols = [_id] + [c for c in base_cols if c in X_train_full.columns]

            Xtr = X_train_full[keep_cols].copy()
            Xte = X_test_full[keep_cols].copy()

            # Add labs only for M3 (fold-specific imputers)
            if tp == "M3":
                print('### Training imputers')
                imputers = train_lab_imputers(
                    X_train_full=Xtr,
                    dfl=dfl,
                    markers=markers,
                    forbidden=forbidden,
                    day_x=day_x,
                    _id = _id,
                    random_state=seed
                )
                Xtr = apply_lab_logic(Xtr, dfl, _id, imputers, markers, forbidden, day_x)
                Xte = apply_lab_logic(Xte, dfl, _id, imputers, markers, forbidden, day_x)

            # Add duration quantile features for M2/M3 (end-of-surgery and later)
            if tp in ("M2", "M3"):
                print('### Adding quantiles')
                Xtr, Xte, gstats = add_duration_quartile_bins(
                    Xtr, Xte, col="duration_of_surg", new_col="duration_of_surg_qbin"
                )
                duration_cutpoints_rows.append({
                    "fold": fold, "timepoint": tp, "type": "global",
                    "cluster_col": None, "n_train_in_group": None,
                    **gstats
                })

                chop_cols = [c for c in Xtr.columns if c.startswith("chopcluster_")]
                Xtr, Xte, stats_rows = add_duration_quartile_bins_by_chopclusters(
                    Xtr, Xte, cluster_cols=chop_cols,
                    duration_col="duration_of_surg", prefix="duration_of_surg_qbin__"
                )
                for r in stats_rows:
                    duration_cutpoints_rows.append({
                        "fold": fold, "timepoint": tp, "type": "cluster",
                        "cluster_col": r["cluster_col"],
                        "n_train_in_group": r["n_train_in_group"],
                        "q25": r["q25"], "q50": r["q50"], "q75": r["q75"],
                        "mean": r["mean"], "median": r["median"],
                    })

            # Align columns between train/test
            for c in Xtr.columns:
                if c not in Xte.columns:
                    Xte[c] = 0
            Xte = Xte[Xtr.columns]

            # Save M2 raw train/test (no SMOTE) for association analysis later
            if keep_m2_train_test and tp == "M2":
                tr_save = Xtr.copy()
                te_save = Xte.copy()
                tr_save["class"] = y_train.values
                te_save["class"] = y_test.values
                tr_save["fold"] = fold
                te_save["fold"] = fold
                tr_save["split"] = "train"
                te_save["split"] = "test"
                m2_assoc_tables.append(pd.concat([tr_save, te_save], ignore_index=True))

                # Also save per-fold immediately
                per_fold_path = os.path.join(outdir, f"m2_features_raw_fold{fold}_train_test.csv")
                pd.concat([tr_save, te_save], ignore_index=True).to_csv(per_fold_path, index=False)

            # Prepare matrices for modeling
            ids_test = Xte[_id].values
            Xtr_raw = Xtr.drop(columns=[_id]).fillna(0)
            Xte_raw = Xte.drop(columns=[_id]).fillna(0)

            preprocessor = build_preprocessor(list(Xtr_raw.columns))
            Xtr_scaled = preprocessor.fit_transform(Xtr_raw)
            Xte_scaled = preprocessor.transform(Xte_raw)

            feat_names_out = preprocessor.get_feature_names_out()
            Xtr_scaled_df = pd.DataFrame(Xtr_scaled, columns=feat_names_out)
            Xte_scaled_df = pd.DataFrame(Xte_scaled, columns=feat_names_out)

            # (Optional) also export scaled M2 features (no SMOTE) for association
            if keep_m2_train_test and tp == "M2":
                tr_scaled = Xtr_scaled_df.copy()
                te_scaled = Xte_scaled_df.copy()
                tr_scaled.insert(0, _id, Xtr[_id].values)
                te_scaled.insert(0, _id, Xte[_id].values)
                tr_scaled["class"] = y_train.values
                te_scaled["class"] = y_test.values
                tr_scaled["fold"] = fold
                te_scaled["fold"] = fold
                tr_scaled["split"] = "train"
                te_scaled["split"] = "test"
                per_fold_scaled_path = os.path.join(outdir, f"m2_features_scaled_fold{fold}_train_test.csv")
                pd.concat([tr_scaled, te_scaled], ignore_index=True).to_csv(per_fold_scaled_path, index=False)

            # Oversample train only
            print('### SMOTE')
            smote = SMOTE(random_state=seed)
            Xtr_bal, ytr_bal = smote.fit_resample(Xtr_scaled_df, y_train)

            # Run each architecture
            for arch in archs:
                print('### Training',str(arch))
                model = make_model(arch, seed=seed)
                model.fit(Xtr_bal, ytr_bal)

                # predict
                y_prob = model.predict_proba(Xte_scaled_df)[:, 1]
                met = compute_binary_metrics(y_test.values, y_prob, threshold=0.5)

                metrics_rows.append({
                    "fold": fold,
                    "timepoint": tp,
                    "arch": arch,
                    "n_train": int(len(train_idx)),
                    "n_test": int(len(test_idx)),
                    **met
                })

                # store per-row predictions (oof)
                fold_pred_df = pd.DataFrame({
                    f"{_id}": ids_test,
                    "fold": fold,
                    "timepoint": tp,
                    "arch": arch,
                    "y_true": y_test.values,
                    "y_prob": y_prob,
                })
                preds_rows.append(fold_pred_df)

                # SHAP (default: M2 + xgb)
                if do_shap and (tp == shap_timepoint) and (arch == shap_arch):
                    if not _HAS_SHAP:
                        raise RuntimeError("shap is not installed, but --do-shap was set.")
                    explainer = shap.TreeExplainer(model)
                    shap_values = explainer.shap_values(Xte_scaled_df)

                    # FIX FOR RF STARTS HERE
                    if isinstance(shap_values, list):
                        # Old SHAP behavior: list of arrays (one per class)
                        shap_mat = shap_values[1] if len(shap_values) > 1 else shap_values[0]
                    elif getattr(shap_values, "ndim", 0) == 3:
                        # New SHAP behavior for RF: (n_samples, n_features, n_classes)
                        # We select index 1 for the positive class
                        shap_mat = shap_values[:, :, 1]
                    else:
                        # Standard behavior (e.g. XGBoost): (n_samples, n_features)
                        shap_mat = shap_values
                    # FIX ENDS HERE

                    s = pd.DataFrame(shap_mat, columns=Xte_scaled_df.columns)
                    s.insert(0, f"{_id}", ids_test)
                    s.insert(1, "fold", fold)
                    s.insert(2, "timepoint", tp)
                    s.insert(3, "arch", arch)
                    shap_rows.append(s)

                    # global importance for this fold
                    imp = np.abs(shap_mat).mean(axis=0)
                    for fname, val in zip(Xte_scaled_df.columns, imp):
                        shap_importance_rows.append({
                            "fold": fold, "timepoint": tp, "arch": arch,
                            "feature": fname, "mean_abs_shap": float(val)
                        })

    # Write outputs
    print('# Writing metrics')
    metrics_df = pd.DataFrame(metrics_rows)
    metrics_df.to_csv(os.path.join(outdir, "cv_metrics_per_fold.csv"), index=False)
    summarize_cv_metrics(metrics_df, ["timepoint", "arch"]).to_csv(
        os.path.join(outdir, "cv_metrics_summary_mean_std.csv"), index=False
    )

    preds_df = pd.concat(preds_rows, ignore_index=True)
    preds_df.to_csv(os.path.join(outdir, "oof_predictions_all.csv"), index=False)

    # Duration quantile cutpoints
    print('# Writing quantiles')
    pd.DataFrame(duration_cutpoints_rows).to_csv(
        os.path.join(outdir, "duration_quantile_cutpoints_per_fold.csv"),
        index=False
    )

    # M2 association export (raw)
    print('# Writing precomputed data')
    if keep_m2_train_test and m2_assoc_tables:
        m2_all = pd.concat(m2_assoc_tables, ignore_index=True)
        m2_all.to_csv(os.path.join(outdir, "m2_features_raw_all_folds_train_test.csv"), index=False)

    ##########################################################################
    # Calibration by procedure cluster (M2, XGB) using out-of-fold predictions
    # Merge predictions with chopcluster cols from base cohort
    chop_cols_all = [c for c in df_all.columns if c.startswith("chopcluster_")]
    if chop_cols_all:
        oof_m2_xgb = preds_df[(preds_df["timepoint"] == "M2") & (preds_df["arch"] == "rf")].copy()
        # attach chopcluster columns
        chop_map = df_all[[_id] + chop_cols_all]
        oof_m2_xgb = oof_m2_xgb.merge(chop_map, on=_id, how="left")
    
        calib_points, cluster_summary = compute_cluster_calibration(oof_m2_xgb, chop_cols_all, n_bins=10)
        calib_points.to_csv(os.path.join(outdir, "m2_rf_calibration_points_by_cluster.csv"), index=False)
        cluster_summary.to_csv(os.path.join(outdir, "m2_rf_calibration_brier_bias_by_cluster.csv"), index=False)

    ##########################################################################
    # SHAP outputs (if enabled)
        print('# SHAP analysis')
    if do_shap and shap_rows:
        shap_all = pd.concat(shap_rows, ignore_index=True)
        shap_all.to_csv(os.path.join(outdir, f"shap_values_{shap_timepoint}_{shap_arch}_testfolds.csv"), index=False)

        imp_df = pd.DataFrame(shap_importance_rows)
        imp_df.to_csv(os.path.join(outdir, f"shap_mean_abs_importance_{shap_timepoint}_{shap_arch}_per_fold.csv"), index=False)
        # aggregate importance
        (imp_df.groupby(["timepoint", "arch", "feature"])["mean_abs_shap"]
              .mean()
              .reset_index()
              .sort_values("mean_abs_shap", ascending=False)
              .to_csv(os.path.join(outdir, f"shap_mean_abs_importance_{shap_timepoint}_{shap_arch}_mean.csv"), index=False)
        )

In [11]:
BASE_PATH = ".."
OUTPUT_DIR = "output.pdp_reproducability_cv"
COL_ID = "_id"
DAY_X = 2
RANDOM_STATE = 42
N_SPLITS = 5

# Ensure output directory exists
os.makedirs(os.path.join(BASE_PATH, OUTPUT_DIR), exist_ok=True)

# -----------------------------------------------------------------------------
# Run the full reproducibility pipeline (M1/M2/M3 × logreg/rf/xgb, CV+SMOTE),
# write all CSVs, and export M2 train/test (NO SMOTE) for association analysis.
# -----------------------------------------------------------------------------

run_all(
    base_path=BASE_PATH,
    outdir=os.path.join(BASE_PATH, OUTPUT_DIR),
    day_x=DAY_X,
    _id = COL_ID,
    n_splits=N_SPLITS,
    seed=RANDOM_STATE,
    do_shap=True,              # set False if you don't want SHAP / shap isn't installed
    shap_timepoint="M2",       # SHAP on M2 like in the manuscript
    shap_arch="rf",
    keep_m2_train_test=True    # <-- keeps train/test (no SMOTE) for association analysis
)

print(f"DONE. Outputs written to: {os.path.join(BASE_PATH, OUTPUT_DIR)}")

## Loading and preprocessing data
>>>FINAL COHORT: 2828
## Fold 1
### Model M1
### SMOTE
### Training logreg
### Training rf
### Training xgb
### Model M2
### Adding quantiles
### SMOTE
### Training logreg
### Training rf
### Training xgb
### Model M3
### Training imputers
### Adding quantiles
### SMOTE
### Training logreg
### Training rf
### Training xgb
## Fold 2
### Model M1
### SMOTE
### Training logreg
### Training rf
### Training xgb
### Model M2
### Adding quantiles
### SMOTE
### Training logreg
### Training rf
### Training xgb
### Model M3
### Training imputers
### Adding quantiles
### SMOTE
### Training logreg
### Training rf
### Training xgb
## Fold 3
### Model M1
### SMOTE
### Training logreg
### Training rf
### Training xgb
### Model M2
### Adding quantiles
### SMOTE
### Training logreg
### Training rf
### Training xgb
### Model M3
### Training imputers
### Adding quantiles
### SMOTE
### Training logreg
### Training rf
### Training xgb
## Fold 4
### Model M1
### SMOTE
### T